# Self-excited instability of a Rijke tube

A **Rijke tube** is the textbook thermoacoustic oscillator: a duct with a heat source
that, for the right time lag, sustains a loud tone all by itself.  It is the simplest
system in which an *active* element -- a flame whose heat release responds to the
acoustic field -- pumps energy into a duct mode and drives it unstable.

The mechanism is **Rayleigh's criterion**: when the unsteady heat release $q'$ is in
phase with the pressure fluctuation $p'$ at the flame, the cycle gains energy and the
oscillation grows.  We model the flame response with a **flame transfer function**
(FTF) -- here the classic $n$--$\tau$ law

$$ \frac{q'(\omega)}{\bar q} \;=\; n\,e^{-i\omega\tau}\,\frac{u'_{\mathrm{ref}}(\omega)}{\bar u_{\mathrm{ref}}}, $$

the heat release responding to the velocity an instant $\tau$ earlier, with gain $n$.

In FNS this is the **source face** $\mathbf S(\omega)$ of the perturbation operator
$\mathbf A(\omega) = \mathbf J_{\mathrm{alg}} + i\omega\mathbf M + \mathbf P(\omega) +
\mathbf S(\omega)$ (theory.md §12.4).  The mean flame is acoustically passive; attaching
a `DynamicSource` makes $\mathbf A$ non-self-adjoint, and the **stability eigenproblem**
$\det\mathbf A(\omega)=0$ acquires complex roots -- modal frequency $\mathrm{Re}\,\omega/2\pi$
and growth rate $-\mathrm{Im}\,\omega$.

We will (1) build the mean flow, (2) attach an $n$--$\tau$ flame, (3) find the unstable
mode, (4) map the stability boundary against the **analytical** compact-flame dispersion
relation, and (5) repeat with the **reacting (equilibrium) flame** driven by the same FTF.

In [ ]:
import os, sys, warnings

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "fns")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from fns.elements import catalog as cat
from fns.elements.dynamic_source import n_tau, n_tau_flame
from fns.perturbation.boundary_bc import PerturbationBC
from fns.perturbation import eigenmodes
from fns.plotting import use_fns_theme, plot_transfer_function
from fns.solver import solve
from fns.solver.control import states_table
from fns.derive import ES_RHO, ES_C, ES_U, ES_P, ES_T, ES_MDOT
from fns.thermo.configure import perfect_gas

use_fns_theme()

## 1. The mean flow

Cold air enters a hard-walled (closed, $u'=0$) inlet, flows down a cold duct of length
$L_1$, crosses a compact flame, and leaves through a hot duct of length $L_2$ to an open
end ($p'=0$).  We keep the mean Mach number low so the acoustics are clean.

The flame is a **perfect-gas heat-release flame** raising the total temperature by
$\Delta T = \dot Q/(\dot m c_p)$.  A `DynamicSource` (the $n$--$\tau$ FTF) is attached to
it; the steady solve ignores it (a constant mean source is acoustically passive), so the
mean flow is the same with or without it.

In [ ]:
R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)
AREA = 0.01                 # tube cross-section [m^2]
L1, L2 = 0.6, 0.4           # cold (upstream) and hot (downstream) duct lengths [m]
MDOT = 0.006                # low mean Mach -> the zero-Mach analytic model is accurate
DT_RISE = 400.0             # mean temperature rise across the flame [K]
QDOT = MDOT * CP * DT_RISE  # heat-release power [W]
REF_EDGE = 1                # velocity reference: the edge just upstream of the flame

def rijke_pg(n, tau):
    # cold air -> duct -> n-tau heat-release flame -> duct -> open end
    flame = cat.heat_release_flame(QDOT, dynamic_source=n_tau_flame(n, tau, ref_edge=REF_EDGE))
    els = [
        cat.mass_flow_inlet(MDOT, 300.0, perturbation_bc=PerturbationBC.hard_wall()),
        cat.duct(L1),
        flame,
        cat.duct(L2),
        cat.pressure_outlet(1.0e5, perturbation_bc=PerturbationBC.open_end()),
    ]
    edges = [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA)]
    prob = cat.build_problem(perfect_gas(R, GAMMA), els, edges, mdot_ref=MDOT, p_ref=1e5, h_ref=CP * 300.0)
    res = solve(prob)
    assert res.converged
    return prob, res.x

prob0, x0 = rijke_pg(0.0, 0.0)
est = states_table(prob0, x0)
print(f"cold (edge 1):  T = {est[ES_T,1]:6.1f} K   c = {est[ES_C,1]:6.1f} m/s   M = {est[ES_U,1]/est[ES_C,1]:.4f}")
print(f"hot  (edge 2):  T = {est[ES_T,2]:6.1f} K   c = {est[ES_C,2]:6.1f} m/s   M = {est[ES_U,2]/est[ES_C,2]:.4f}")

## 2. The flame transfer function

The $n$--$\tau$ model is a pure gain $n$ and a delay $\tau$: a flat magnitude and a phase
that winds linearly with frequency.  The new `plot_transfer_function` helper reads any
complex function of frequency as magnitude over phase (or as a Nyquist diagram).

In [ ]:
N, TAU = 0.8, 4.0e-3       # gain and time lag [s]
ftf = n_tau(N, TAU)
freqs = np.linspace(0.0, 400.0, 400)

plot_transfer_function(
    ftf, freqs, names=[f"n-tau  (n={N}, tau={TAU*1e3:.1f} ms)"],
    title="Flame transfer function  q'/q-bar  /  (u'/u-bar)",
).show()

## 3. The unstable mode

`eigenmodes` solves the nonlinear eigenproblem $\det\mathbf A(\omega)=0$ by Beyn's
contour-integral method, returning each mode's frequency and growth rate.  We search the
isentropic (no convected entropy) operator -- the standard acoustic-network-with-a-flame
model -- so the flame's energy jump stays physical while entropy convection is dropped.

A **positive growth rate** is a self-excited (unstable) mode.

In [ ]:
prob, x = rijke_pg(N, TAU)
res = eigenmodes(prob, x, freq_band=(40.0, 320.0), growth_band=(-200.0, 200.0), isentropic=True)

for m in res.summary():
    flag = "  <-- UNSTABLE" if m["unstable"] else ""
    print(f"f = {m['freq_hz']:7.2f} Hz    growth = {m['growth_rate']:+8.2f} 1/s{flag}")

res.plot_spectrum(title=f"Rijke spectrum  (n={N}, tau={TAU*1e3:.0f} ms)").show()

## 4. The stability map -- and analytical verification

Sweeping the time lag $\tau$ traces the classic $n$--$\tau$ stability band: the peak
growth rate rises and falls, crossing zero into instability for a range of $\tau$.

We check FNS against the **analytical compact-flame dispersion relation** -- a two-region
(cold/hot) acoustic tube with the zero-mean-Mach jump

$$ p'_1 = p'_2, \qquad u'_2 - u'_1 = \frac{\gamma-1}{\gamma\,\bar p\,A}\,q', $$

whose $\det = 0$ gives the complex modes.  At this low Mach the two agree closely; the
small residual is the $O(M)$ mean-flow damping the zero-Mach analytic omits.

In [ ]:
def region_means(prob, x):
    est = states_table(prob, x)
    return dict(rho1=est[ES_RHO, 1], c1=est[ES_C, 1], u1=est[ES_U, 1], p1=est[ES_P, 1],
                rho2=est[ES_RHO, 2], c2=est[ES_C, 2])

def analytic_det(omega, m, Qdot, n, tau):
    k1, k2 = omega / m['c1'], omega / m['c2']
    Z1, Z2 = m['rho1'] * m['c1'], m['rho2'] * m['c2']
    eP1, eM1 = np.exp(1j * k1 * L1), np.exp(-1j * k1 * L1)
    eP2, eM2 = np.exp(-1j * k2 * L2), np.exp(1j * k2 * L2)
    Th = (GAMMA - 1.0) * Qdot / (GAMMA * m['p1'] * AREA) * (n * np.exp(-1j * omega * tau)) / (Z1 * m['u1'])
    M = np.array([[eP1, -eM1, 0, 0],
                  [1, 1, -1, -1],
                  [-(1 / Z1 + Th), (1 / Z1 + Th), 1 / Z2, -1 / Z2],
                  [0, 0, eP2, eM2]], dtype=complex)
    return np.linalg.det(M)

def analytic_modes(m, Qdot, n, tau, fband=(40.0, 320.0)):
    wr = 2 * np.pi * np.linspace(*fband, 60)
    wi = np.linspace(-220.0, 220.0, 60)
    WR, WI = np.meshgrid(wr, wi)
    D = np.abs(np.vectorize(lambda w: analytic_det(w, m, Qdot, n, tau))(WR + 1j * WI))
    grid = WR + 1j * WI
    seeds = [grid[i, j] for i in range(1, D.shape[0] - 1) for j in range(1, D.shape[1] - 1)
             if D[i, j] == D[i - 1:i + 2, j - 1:j + 2].min()]
    out = []
    for w0 in seeds:
        w = complex(w0)
        for _ in range(60):
            h = 1e-3 * (abs(w) + 1.0)
            d = analytic_det(w, m, Qdot, n, tau)
            dp = (analytic_det(w + h, m, Qdot, n, tau) - analytic_det(w - h, m, Qdot, n, tau)) / (2 * h)
            if dp == 0:
                break
            s = d / dp
            w -= s
            if abs(s) < 1e-9:
                break
        if 2 * np.pi * fband[0] < w.real < 2 * np.pi * fband[1] and -220.0 < w.imag < 220.0:
            if all(abs(w - o) > 1e-1 for o in out):
                out.append(w)
    return out

m_mean = region_means(*rijke_pg(0.0, 0.0))
taus = np.linspace(0.0, 6.0e-3, 25)
g_fns, g_an = [], []
for tau in taus:
    pr, xx = rijke_pg(N, tau)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r = eigenmodes(pr, xx, freq_band=(40.0, 320.0), growth_band=(-260.0, 260.0), isentropic=True)
    g_fns.append(max(r.growth_rates) if len(r) else np.nan)
    am = analytic_modes(m_mean, QDOT, N, tau)
    g_an.append(max((-w.imag for w in am), default=np.nan))

fig = go.Figure()
fig.add_scatter(x=taus * 1e3, y=g_an, mode="lines", name="analytic")
fig.add_scatter(x=taus * 1e3, y=g_fns, mode="markers", name="FNS eigenmodes", marker=dict(size=8))
fig.add_hline(y=0.0, line_dash="dash", annotation_text="stability boundary")
fig.update_layout(title="Rijke stability map: peak growth rate vs. flame time lag",
                  xaxis_title="time lag  tau  (ms)", yaxis_title="growth rate  (1/s)", template="fns")
fig.show()

The FNS markers sit on the analytic curve and cross the stability boundary at the same
lags -- the sign **and** magnitude of the heat-release coupling check out: the dynamic
source genuinely drives the instability, not merely perturbs it.

## 5. The reacting flame

The same FTF drives the **reacting equilibrium flame** -- the source is flame-agnostic.
Here the mean flame temperature is not prescribed: stoichiometric H$_2$/air enters frozen
and burns to chemical equilibrium across the flame (the reacting mean solver), and the
mean heat release $\bar Q$ for the FTF auto-derives from the converged temperature rise.

The reacting acoustics use the gas's **actual caloric coupling** in the characteristic
maps -- the per-edge $(\partial h/\partial\rho)_p,\ (\partial h/\partial p)_\rho$ from the
equilibrium closure, *not* the perfect-gas $c_p/R$ -- so the reacting case is now
**quantitatively** verified, not just demonstrated.  Two physical differences from the
fixed-power perfect-gas flame are worth noting:

* The equilibrium flame **conserves total enthalpy** (the heat release is internal, in the
  absolute-enthalpy datum), so its compact jump is *mass-flux continuous* ($\rho u'$
  continuous), not velocity-continuous.
* That gives the flame an **intrinsic quasi-steady response** (its heat release tracks the
  reactant supply), so the effective FTF is $1 + n\,e^{-i\omega\tau}$ -- it takes a larger
  lag to destabilize than the perfect-gas flame.

We verify against the matching **equilibrium-flame dispersion relation**: five waves (the
two acoustic + an entropy spot shed at the flame) with the exact equilibrium EOS
derivatives and the same source coupling FNS stamps.

In [ ]:
from thermolib import ThermoInp, Thermo
from fns.composition import resolve_composition, enthalpy_mass
from fns.thermo.configure import equilibrium
from fns.thermo.api import EQ_FROZEN, EQ_KERNEL

THERMO_INP = os.path.join(_root, "thermolib", "data", "thermo.inp")
lib = ThermoInp(THERMO_INP).library(["H2", "O2", "N2", "H2O", "OH", "H", "O", "HO2", "NO"])
gas = Thermo(lib)
fuel_air = {"H2": 1.0, "O2": 0.5, "N2": 0.5 * 3.76}   # stoichiometric, mole basis
Y, Z = resolve_composition(lib, fuel_air, basis="mole")
H_REACT = enthalpy_mass(lib, Y, 300.0)
MDOT_R = 0.02

def rijke_reacting(n, tau):
    els = [
        cat.mass_flow_inlet(MDOT_R, 300.0, composition=fuel_air, basis="mole",
                            perturbation_bc=PerturbationBC.hard_wall()),
        cat.duct(L1),
        cat.equilibrium_flame(dynamic_source=n_tau_flame(n, tau, ref_edge=REF_EDGE)),
        cat.duct(L2),
        cat.pressure_outlet(1.0e5, Tt_backflow=300.0, composition=fuel_air, basis="mole",
                            perturbation_bc=PerturbationBC.open_end()),
    ]
    edges = [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA)]
    edge_models = [EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]   # unburnt approach -> burnt products
    prob = cat.build_problem(equilibrium(gas.mech), els, edges, mdot_ref=MDOT_R, p_ref=1e5,
                             h_ref=H_REACT, edge_models=edge_models)
    res = solve(prob)
    assert res.converged
    return prob, res.x

prob_r, x_r = rijke_reacting(0.0, 0.0)
est_r = states_table(prob_r, x_r)
print(f"unburnt: T = {est_r[ES_T,1]:7.1f} K     burnt: T = {est_r[ES_T,2]:7.1f} K  (adiabatic flame)")

In [ ]:
from fns.thermo.api import thermo_state


def reacting_caloric(prob, x, est, e):
    # (dh/drho)_p, (dh/dp)_rho at edge e via a complex step of the equilibrium closure
    xi = np.ascontiguousarray(x[3:3 + prob.n_elem, e]).astype(np.complex128)
    ht, p, d = complex(x[2, e]), float(est[ES_P, e]), 1e-30
    mid = int(prob.edge_model[e])
    drho_dh = thermo_state(mid, prob.tf, prob.ti, xi, ht + 1j * d, complex(p))[1].imag / d
    drho_dp = thermo_state(mid, prob.tf, prob.ti, xi, ht, p + 1j * d)[1].imag / d
    return 1.0 / drho_dh, -drho_dp / drho_dh


def reacting_det(omega, mm, cal1, cal2, delta, n, tau):
    # 5 waves [f1, g1, f2, g2, h2]: mass-flux + momentum + (h_t cont + source), equilibrium EOS
    rho1, cc1, u1, p1, rho2, cc2, u2, p2 = mm
    a1, b1 = cal1
    a2, b2 = cal2
    Z1, Z2 = rho1 * cc1, rho2 * cc2
    F = n * np.exp(-1j * omega * tau)
    k1p, k1m, k2p, k2m = omega / (u1 + cc1), omega / (cc1 - u1), omega / (u2 + cc2), omega / (cc2 - u2)
    M = np.zeros((5, 5), complex)
    M[0, 0], M[0, 1] = np.exp(1j * k1p * L1), -np.exp(-1j * k1m * L1)        # inlet hard wall
    M[1, 2], M[1, 3] = np.exp(-1j * k2p * L2), np.exp(1j * k2m * L2)         # outlet open end
    M[2] = [AREA * (u1 * Z1 / cc1**2 + rho1), AREA * (u1 * Z1 / cc1**2 - rho1),
            -AREA * (u2 * Z2 / cc2**2 + rho2), -AREA * (u2 * Z2 / cc2**2 - rho2), -AREA * u2]   # mass flux
    M[3] = [Z1 + u1**2 * Z1 / cc1**2 + 2 * rho1 * u1, Z1 + u1**2 * Z1 / cc1**2 - 2 * rho1 * u1,
            -(Z2 + u2**2 * Z2 / cc2**2 + 2 * rho2 * u2), -(Z2 + u2**2 * Z2 / cc2**2 - 2 * rho2 * u2), -u2**2]  # momentum
    ht1, ht2, src = a1 * Z1 / cc1**2 + b1 * Z1, a2 * Z2 / cc2**2 + b2 * Z2, delta * F / u1
    M[4] = [-ht1 - src, -ht1 + src, ht2, ht2, a2]                            # energy + heat-release source
    return np.linalg.det(M)


def reacting_mode(mm, cal1, cal2, delta, n, tau, seed):
    w = complex(seed)
    for _ in range(200):
        h = 1e-3 * (abs(w) + 1.0)
        dp = (reacting_det(w + h, mm, cal1, cal2, delta, n, tau) - reacting_det(w - h, mm, cal1, cal2, delta, n, tau)) / (2 * h)
        if dp == 0:
            break
        s = reacting_det(w, mm, cal1, cal2, delta, n, tau) / dp
        w -= s
        if abs(s) < 1e-9:
            break
    return w.real / (2 * np.pi), -w.imag


def cp_eff(col):  # gamma R / (gamma - 1) from the mean state (sound-speed consistent)
    g = col[ES_RHO] * col[ES_C] ** 2 / col[ES_P]
    return g * (col[ES_P] / (col[ES_RHO] * col[ES_T])) / (g - 1.0)


# the same n-tau response, now on the reacting flame (a larger lag, since the equilibrium
# flame's intrinsic quasi-steady response must be overcome)
TAU_R = 6.0e-3
prob_r, x_r = rijke_reacting(N, TAU_R)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    res_r = eigenmodes(prob_r, x_r, freq_band=(50.0, 200.0), growth_band=(-250.0, 250.0), isentropic=True)
for mm in res_r.summary():
    flag = "  <-- UNSTABLE" if mm["unstable"] else ""
    print(f"f = {mm['freq_hz']:7.2f} Hz    growth = {mm['growth_rate']:+8.2f} 1/s{flag}")

# quantitative check against the equilibrium-flame dispersion relation
est_r = states_table(prob_r, x_r)
means = (est_r[ES_RHO, 1], est_r[ES_C, 1], est_r[ES_U, 1], est_r[ES_P, 1],
         est_r[ES_RHO, 2], est_r[ES_C, 2], est_r[ES_U, 2], est_r[ES_P, 2])
delta = 0.5 * (cp_eff(est_r[:, 1]) + cp_eff(est_r[:, 2])) * (est_r[ES_T, 2] - est_r[ES_T, 1])
cal1, cal2 = reacting_caloric(prob_r, x_r, est_r, 1), reacting_caloric(prob_r, x_r, est_r, 2)
f_fns, g_fns = max(zip(res_r.freqs, res_r.growth_rates), key=lambda t: t[1])
f_an, g_an = reacting_mode(means, cal1, cal2, delta, N, TAU_R, 2 * np.pi * f_fns)
print(f"\nmost-unstable mode:  FNS       f = {f_fns:6.2f} Hz, growth = {g_fns:+7.2f} 1/s")
print(f"                     analytic  f = {f_an:6.2f} Hz, growth = {g_an:+7.2f} 1/s")

res_r.plot_spectrum(title=f"Reacting Rijke spectrum  (H2/air, n={N}, tau={TAU_R*1e3:.0f} ms)").show()

## Summary

* The dynamic **source face** $\mathbf S(\omega)$ -- a flame transfer function or a
  fluctuating mass injection -- drops into the same operator $\mathbf A(\omega)$ the
  scattering and stability drivers already use; no new solver.
* For the heat release it lands on the **downstream energy row** with the verified sign,
  making the operator active; `eigenmodes` then finds the self-excited modes.
* FNS reproduces the **analytical** Rijke dispersion relation in frequency and growth
  rate, including the stability boundary -- the implementation is quantitatively correct.
* The **reacting** equilibrium flame is driven by the identical FTF, with its mean heat
  release taken from the reacting solve.  Because the perturbation maps use the gas's
  actual per-edge caloric coupling, the reacting case matches its own
  equilibrium-flame dispersion relation in frequency **and** growth rate -- the reacting
  Rijke is quantitatively verified, not just demonstrated.